# 1. GDPR overview
Focus on lawful processing and the key principles with concrete examples. Start by building your own short summary from the official EUR-Lex GDPR overview page, then use that summary as your reference sheet for the rest of the notebook.

In [ ]:
# Imports
import hashlib
import hmac
from datetime import UTC, datetime, timedelta
from pprint import pprint
from typing import TypedDict

# Fixed reference time keeps the teaching output reproducible.
DEMO_NOW = datetime(2025, 1, 15, 9, 0, tzinfo=UTC)
PSEUDONYMIZATION_KEY = b"gdpr-demo-key-v1"

## 1.1 Learning goals
- Identify key GDPR actors (data subject, controller, processor) and records (RoPA (Record of Processing Activities), DPIA (Data Protection Impact Assessment)).
- Know the six lawful bases and valid consent criteria.
- Apply the seven principles to simple processing steps.
- Identify data subject rights and response timelines.
- Practice data minimization and purpose limitation in code.

Use the prompts to reason about lawful processing, minimization, retention, and accountability.

### First task: summarize the official GDPR overview
Before continuing, read the EUR-Lex summary page: [General data protection regulation (GDPR)](https://eur-lex.europa.eu/en/legal-content/summary/general-data-protection-regulation-gdpr.html).

Write a short summary in your own words with **5-8 bullet points** covering at least:
- the aim of the GDPR,
- the main individual rights it highlights,
- the key rules for businesses and organisations,
- the role of supervisory authorities / the one-stop shop,
- and the date from which the GDPR applies.

Use that page as your **working reference** throughout the rest of this notebook. For later questions, start from this summary page first, then drill into the full GDPR text when you need article-level detail.

## 1.2 Quick reference
- Actors: data subject (individual), controller (decides purposes/means), processor (acts for the controller), DPO (independent advisor).
- Records: RoPA (Record of Processing Activities), DPIA (Data Protection Impact Assessment — an impact assessment for higher-risk processing), DSR log (data subject requests).
- International transfers: rely on adequacy, SCCs, or BCRs; document the transfer risk assessment.
- Working lecture reference: [EUR-Lex GDPR summary page](https://eur-lex.europa.eu/en/legal-content/summary/general-data-protection-regulation-gdpr.html).

![First page of the GDPR regulation](assets/gdpr-regulation-cover.png)

*This primary legal document gives context for the principles explored in this notebook. Use the official [EUR-Lex text](https://eur-lex.europa.eu/eli/reg/2016/679/oj) for the authoritative wording.*

Source: [GDPR p1 on Wikimedia Commons](https://commons.wikimedia.org/wiki/File:GDPR_p1.svg), the Official Journal of the European Union (public-domain legal publication).

## 1.3 Lawful bases (Article 6)
1. **Consent**: freely given, specific, informed, unambiguous; withdrawal as easy as acceptance.
2. **Contract**: necessary to perform or prepare a contract with the data subject.
3. **Legal obligation**: required to comply with a legal duty.
4. **Vital interests**: necessary to protect someone's life.
5. **Public task**: carried out in the public interest or under official authority.
6. **Legitimate interests**: controller's interests outweigh risks without overriding rights; document the assessment.

## 1.4 Enforcement and penalties
GDPR infringements can create financial, operational, and reputational consequences. The maximum administrative fine depends on the obligation breached:

- **Lower tier:** up to **EUR 10 million**, or, for an undertaking, up to **2% of its total worldwide annual turnover from the preceding financial year**, whichever is higher. This tier covers obligations including records of processing, data protection by design and by default, processor arrangements, security, breach notification, and DPIAs.
- **Higher tier:** up to **EUR 20 million**, or, for an undertaking, up to **4% of its total worldwide annual turnover from the preceding financial year**, whichever is higher. This tier covers infringements including the core principles, lawful-basis and consent requirements, special-category data rules, data-subject rights, and international-transfer requirements.

These are statutory ceilings, not automatic price tags. Supervisory authorities must consider factors such as the infringement's nature, gravity, duration, intent, mitigation, cooperation, affected data, and previous infringements when deciding whether to impose a fine and how large it should be.

Authorities can also use **corrective powers** beyond fines, including warnings, reprimands, orders to comply, restrictions on processing, and temporary or definitive processing bans. For a PM, those operational measures may be more immediate than the fine: a feature can be delayed, restricted, redesigned, or stopped.

Sources: [GDPR Article 83 — general conditions for administrative fines](https://eur-lex.europa.eu/eli/reg/2016/679/art_83/oj) and [GDPR Article 58 — supervisory authority powers](https://eur-lex.europa.eu/eli/reg/2016/679/art_58/oj).

## 2. Article 5: 7 principles of GDPR (summary)
1. **Lawfulness, fairness, transparency**: process data legally, fairly, and in an understandable way for the data subject.
2. **Purpose limitation**: collect for specific, explicit, legitimate purposes; avoid incompatible reuse.
3. **Data minimization**: keep only data that is adequate, relevant, and necessary.
4. **Accuracy**: keep data accurate and up to date; rectify or erase without delay when needed.
5. **Storage limitation**: keep data identifiable only as long as necessary for the purposes.
6. **Integrity and confidentiality**: protect against unauthorized access, loss, or disclosure.
7. **Accountability**: be able to prove compliance with the principles.

### 2.1 Key articles and sources
- Start here for the rest of this notebook: [EUR-Lex GDPR summary page](https://eur-lex.europa.eu/en/legal-content/summary/general-data-protection-regulation-gdpr.html)
- Lawfulness, fairness, transparency — Article 5(1)(a): [Eur-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj)
- Purpose limitation — Article 5(1)(b): [Eur-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj)
- Data minimization — Article 5(1)(c): [Eur-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj)
- Accuracy — Article 5(1)(d): [Eur-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj)
- Storage limitation — Article 5(1)(e): [Eur-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj)
- Integrity and confidentiality — Article 5(1)(f): [Eur-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj)
- Accountability — Article 5(2): [Eur-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj)

Use the summary page for the big picture and the full GDPR text when you need exact article wording.

## 3. Applying the principles in code
Each subsection exercises a principle with multiple cases so you can see the outputs.

### 3.1 Lawfulness, fairness, transparency
This snippet demonstrates the idea: **don't process unless you have (a) a lawful basis and (b) you informed the person**.

What the code does:
- Defines a fixed set of allowed lawful bases (`lawful_bases`).
- Defines `can_process(...)` that returns `True` only if the lawful basis is recognized *and* a notice was sent.
- Runs several scenarios and prints a table-like list of results (`can_process: True/False`).

**Teaching boundary:** recognizing a lawful-basis label and sending a notice are necessary checks, but they do not prove lawfulness. A real assessment must also test necessity, purpose, fairness, proportionality, and any basis-specific conditions.


In [ ]:
# Ensures a lawful basis is selected and a notice was sent before processing proceeds.
# Lawfulness, fairness, transparency: require a lawful basis and a clear notice before processing
lawful_bases = {
    "consent",
    "contract",
    "legal_obligation",
    "vital_interests",
    "public_task",
    "legitimate_interests",
}


class Scenario(TypedDict):
    purpose: str
    lawful_basis: str
    notice_sent: bool


def can_process(lawful_basis: str, notice_sent: bool) -> bool:
    return lawful_basis in lawful_bases and notice_sent


scenarios: list[Scenario] = [
    {"purpose": "account_signup", "lawful_basis": "consent", "notice_sent": True},
    {"purpose": "order_shipping", "lawful_basis": "contract", "notice_sent": False},
    {
        "purpose": "fraud_monitoring",
        "lawful_basis": "legitimate_interests",
        "notice_sent": True,
    },
    {"purpose": "ad_targeting", "lawful_basis": "unknown", "notice_sent": True},
]
decisions = [
    {
        **scenario,
        "can_process": can_process(scenario["lawful_basis"], scenario["notice_sent"]),
    }
    for scenario in scenarios
]
pprint(decisions)

### 3.2 Purpose limitation
Here we model a very common control: **data collected for one purpose must not silently be reused for another**.

What the code does:
- `allowed_purposes` is a stand-in for what your privacy notice/internal documentation says you do with the data.
- `use_data(purpose)` raises a `ValueError` if the purpose is not allowed.
- The loop catches that error and turns it into a readable decision like `Rejected: Purpose not allowed ...`.

How to interpret the output: a row is allowed if the `decision` starts with `Using data only...`; otherwise it was rejected.


In [ ]:
# Allows data use only for declared purposes; raises an error if a new purpose is not covered.
# Purpose limitation: reject uses outside the documented purpose
allowed_purposes = {"fulfill_order", "fraud_detection"}


def use_data(purpose: str) -> str:
    if purpose not in allowed_purposes:
        raise ValueError("Purpose not allowed by the privacy notice")
    return f"Using data only for {purpose}"


requests = [
    {"purpose": "fulfill_order", "data": "shipping address"},
    {"purpose": "fraud_detection", "data": "device fingerprint"},
    {"purpose": "ad_targeting", "data": "page views"},
]
decisions = []
for req in requests:
    try:
        status = use_data(req["purpose"])
    except ValueError as err:
        status = f"Rejected: {err}"
    decisions.append({**req, "decision": status})
pprint(decisions)

### 3.3 Data minimization
The goal is to create a **reduced analytics dataset** that contains only what’s needed for the stated purpose.

What the code does:
- Starts with user profiles that contain direct identifiers (name/email) and extra attributes (address, marketing opt-in).
- Defines `needed = {'user_id', 'country', 'age'}` as the minimal set required for the analytics question.
- Builds `analytics_view` by filtering each record down to those keys only.

Why this matters: less data stored/shared → smaller breach impact and easier compliance.


In [ ]:
# Builds a reduced view of a record that keeps only fields needed for the analytics purpose.
# Data minimization: keep only fields needed for analytics
users = [
    {
        "user_id": "u-101",
        "name": "Ana",
        "email": "ana@example.com",
        "age": 34,
        "country": "FR",
        "address": "12 Rue Bleue",
        "marketing_opt_in": True,
    },
    {
        "user_id": "u-102",
        "name": "Lee",
        "email": "lee@example.com",
        "age": 28,
        "country": "DE",
        "address": "9 Hauptstrasse",
        "marketing_opt_in": False,
    },
    {
        "user_id": "u-103",
        "name": "Sam",
        "email": "sam@example.com",
        "age": 41,
        "country": "ES",
        "address": "Calle Verde 3",
        "marketing_opt_in": True,
    },
]
needed = {"user_id", "country", "age"}
analytics_view = [{k: v for k, v in user.items() if k in needed} for user in users]
pprint(analytics_view)

### 3.4 Accuracy
This example shows a safe update pattern: **only apply an update if it is newer than what you already have**.

What the code does:
- Keeps a `current` email record with an `updated_at` timestamp.
- Iterates through `incoming_updates` and applies `update_if_newer(...)`.
- Builds a `history` list showing whether each incoming update changed the current value.

How to interpret the output: `updated: True` means the incoming timestamp was newer; `False` means it was ignored as stale.


In [ ]:
# Updates a record only when the incoming data is newer to keep information accurate.
# Accuracy: accept updates only when the incoming record is newer
class EmailUpdate(TypedDict):
    email: str
    updated_at: datetime


current: EmailUpdate = {
    "email": "ana@old.com",
    "updated_at": datetime.fromisoformat("2024-01-01T12:00:00+00:00"),
}
incoming_updates: list[EmailUpdate] = [
    {
        "email": "ana@example.com",
        "updated_at": datetime.fromisoformat("2024-02-01T09:00:00+00:00"),
    },
    {
        "email": "ana@typo.com",
        "updated_at": datetime.fromisoformat("2023-12-31T09:00:00+00:00"),
    },
    {
        "email": "ana@work.com",
        "updated_at": datetime.fromisoformat("2024-03-15T18:30:00+00:00"),
    },
]


def update_if_newer(
    current_record: EmailUpdate, new_record: EmailUpdate
) -> EmailUpdate:
    if new_record["updated_at"] > current_record["updated_at"]:
        current_record.update(new_record)
    return current_record


history = []
for update in incoming_updates:
    before = current.copy()
    update_if_newer(current, update)
    history.append(
        {
            "incoming_at": update["updated_at"].isoformat(),
            "incoming_email": update["email"],
            "updated": current != before,
            "current_email": current["email"],
        }
    )
pprint(history)

### 3.5 Storage limitation
This is a simplified retention rule: **delete or de-identify records after a defined time window**.

What the code does:
- Creates sample rows with different `created_at` ages.
- `apply_retention(data, days=30)` computes a cutoff and keeps only records newer than the cutoff.
- Prints a report that includes `age_days` and whether each row was `kept`.

Real-world note: the retention period usually depends on purpose + legal obligations, and should be documented (e.g., retention schedule).


In [ ]:
# Applies a retention window and drops rows older than the allowed cutoff.
# Storage limitation: use a fixed reference time so the demo output stays stable across reruns.
class RetentionRow(TypedDict):
    id: int
    created_at: datetime
    source: str


now = DEMO_NOW
rows: list[RetentionRow] = [
    {"id": 1, "created_at": now - timedelta(days=10), "source": "consent_form"},
    {"id": 2, "created_at": now - timedelta(days=45), "source": "support_ticket"},
    {"id": 3, "created_at": now - timedelta(days=30), "source": "checkout"},
    {"id": 4, "created_at": now - timedelta(days=75), "source": "legacy_migration"},
    {"id": 5, "created_at": now - timedelta(days=5), "source": "signup"},
]


def apply_retention(
    data: list[RetentionRow], days: int = 30, reference_time: datetime | None = None
) -> list[RetentionRow]:
    reference_time = reference_time or DEMO_NOW
    cutoff = reference_time - timedelta(days=days)
    return [row for row in data if row["created_at"] >= cutoff]


kept = apply_retention(rows, days=30, reference_time=now)
kept_ids = {row["id"] for row in kept}
report = [
    {
        "id": row["id"],
        "age_days": (now - row["created_at"]).days,
        "kept": row["id"] in kept_ids,
    }
    for row in rows
]
pprint(report)

### 3.6 Integrity and confidentiality
This section shows two common security controls:
- **Pseudonymization** (reducing direct identifiability when you don’t need raw identifiers), and
- **Role-based access control** (only certain roles can see raw personal data).

What the code does:
- `pseudonymize(...)` generates a short hash-like token for each email (toy version).
- `view_raw(...)` raises `PermissionError` unless the role is allowed.
- Prints pseudonyms plus an access check result for two roles.

Security note: real pseudonymization typically uses a stable keyed hash (e.g., HMAC) and secure key management.


In [ ]:
# Pseudonymizes identifiers with a keyed hash and gates raw access by role.
# Integrity and confidentiality: keep the demo deterministic with a fixed teaching key.
def pseudonymize(value: str, key: bytes = PSEUDONYMIZATION_KEY) -> str:
    return hmac.new(key, value.encode(), hashlib.sha256).hexdigest()[:16]


def view_raw(email: str, role: str) -> str:
    if role not in {"dpo", "security_admin"}:
        raise PermissionError("Raw access denied")
    return email


emails = ["ana@example.com", "lee@example.com", "sam@example.com"]
pseudonyms = {email: pseudonymize(email) for email in emails}
access_checks = []
for role in ["analyst", "security_admin"]:
    try:
        access_checks.append({"role": role, "raw_view": view_raw(emails[0], role)})
    except PermissionError as err:
        access_checks.append({"role": role, "raw_view": str(err)})
pprint({"pseudonyms": pseudonyms, "access_checks": access_checks})

### 3.7 Accountability
Accountability means you must be able to **demonstrate** compliance, not just claim it.

What the code does:
- `log_action(...)` creates an audit-log entry with a timestamp, action name, lawful basis, and subject identifier.
- The loop logs a few example events.
- `pprint(audit_log)` prints the evidence trail you would later rely on during an audit or incident review.

Real-world note: audit logs should be protected (tamper resistance), access-controlled, and have their own retention policy.


In [ ]:
# Adds an auditable log entry with timestamp, action, lawful basis, and subject id.
# Accountability: use explicit event times so reruns keep the same evidence trail.
audit_log = []


def log_action(
    action: str,
    lawful_basis: str,
    subject_id: str,
    event_time: datetime | None = None,
):
    event_time = event_time or DEMO_NOW
    timestamp = event_time.isoformat().replace("+00:00", "Z")
    audit_log.append(
        {
            "timestamp": timestamp,
            "action": action,
            "lawful_basis": lawful_basis,
            "subject_id": subject_id,
        }
    )


events = [
    ("respond_access_request", "consent", "subj-001"),
    ("delete_expired_records", "legal_obligation", "batch-2024-02"),
    ("correct_email", "contract", "subj-002"),
]
for index, (action, basis, subject) in enumerate(events):
    log_action(
        action,
        basis,
        subject_id=subject,
        event_time=DEMO_NOW + timedelta(minutes=index * 15),
    )
pprint(audit_log)

## 4. Data subject rights
Access, rectification, erasure, restriction, portability, and objection are distinct rights with different conditions. Article 22 also provides protections for certain decisions based solely on automated processing that produce legal or similarly significant effects, including safeguards such as human intervention where applicable. Aim to respond to rights requests within one month, with a possible extension for complex or numerous requests. Tie each dataset to an owner, a contact path, and the fields you can retrieve.


### 4.1 Mini-case: is this lawful?
- **Scenario:** A company uses a third-party leave-management platform so employees can request medical leave and upload a doctor's certificate. The employer decides why the data is collected and how it is used in HR decisions; the software vendor hosts the platform and processes the records on the employer's instructions.
- **Context:** The certificates contain health information, the system stores records for all staff across several EU offices, and uploads are automatically classified into leave categories using a new AI-assisted triage feature.
- **Retention / rights context:** National employment-law rules require the employer to keep approved leave records for 3 years, even if an employee later asks for erasure.
- Identify controller and processor roles.
- Select and justify the lawful basis.
- Check for special-category data (Article 9) and applicable conditions.
- Decide if a DPIA is needed (systematic monitoring, large-scale sensitive data, or new tech).
- Note any limits to rights (e.g., erasure blocked by a legal obligation).
Document your reasoning in the next cell.


In [ ]:
# Capture roles, lawful basis, and rights limits for the mini-case.
analysis = {
    "controller": "",
    "processor": "",
    "lawful_basis": "",
    "special_category": False,
    "article_9_condition": "",
    "dpia_needed": "maybe",
    "rights_limits": [],
    "reasoning": "",
}
analysis

### 4.2 Example solution
A reasonable answer is shown below. In practice, some legal details can vary by employer context and national employment law, but this captures the main GDPR analysis for the scenario.

In [ ]:
# Example solution for the mini-case.
analysis_solution = {
    "controller": "The employer, because it decides the HR purpose and means of the leave-management process.",
    "processor": "The leave-management software vendor, because it hosts and handles the records on the employer's instructions.",
    "lawful_basis": "legal_obligation",
    "special_category": True,
    "article_9_condition": "Employment and social-protection obligations / rights under Article 9(2)(b), because the records contain health data used for leave administration.",
    "dpia_needed": "likely yes",
    "rights_limits": [
        "Erasure may be limited while the employer must keep approved leave records for the 3-year statutory retention period.",
        "Access and rectification still apply, but identity verification and narrow scope handling are needed for sensitive HR records.",
    ],
    "reasoning": "Health certificates are special-category data. The employer is the controller and the vendor is the processor. A legal-obligation basis fits the employer's duty to manage and retain leave records, while Article 9 needs an additional condition because health data is involved. A DPIA is likely appropriate because the system processes sensitive data for all staff across several offices and adds a new AI-assisted triage feature, which raises risk and accountability concerns.",
}
analysis_solution

## 5. Worked example: pseudonymization + minimization
The next cell shows a simple pattern you can reuse: derive a stable subject identifier from an email address, then keep only the fields required for the stated analytics purpose. In real systems, prefer keyed hashing (HMAC) or a dedicated pseudonymization service to avoid linkability across datasets.

In [ ]:
# Demonstrates pseudonymizing emails and keeping only the minimal fields for age analytics.
class RawUser(TypedDict):
    name: str
    email: str
    age: int
    city: str
    signup_date: datetime
    marketing_opt_in: bool


raw_users: list[RawUser] = [
    {
        "name": "Ana",
        "email": "ana@example.com",
        "age": 34,
        "city": "Paris",
        "signup_date": datetime(2024, 1, 15, tzinfo=UTC),
        "marketing_opt_in": True,
    },
    {
        "name": "Lee",
        "email": "lee@example.com",
        "age": 28,
        "city": "Berlin",
        "signup_date": datetime(2024, 2, 3, tzinfo=UTC),
        "marketing_opt_in": False,
    },
    {
        "name": "Sam",
        "email": "sam@example.com",
        "age": 41,
        "city": "Lyon",
        "signup_date": datetime(2023, 12, 20, tzinfo=UTC),
        "marketing_opt_in": True,
    },
    {
        "name": "Maya",
        "email": "maya@example.com",
        "age": 22,
        "city": "Madrid",
        "signup_date": datetime(2024, 3, 8, tzinfo=UTC),
        "marketing_opt_in": False,
    },
]


# Suppose the purpose is age-distribution analytics (no need for direct identifiers).
def pseudonymize_email(email: str, key: bytes = PSEUDONYMIZATION_KEY) -> str:
    return hmac.new(key, email.encode(), hashlib.sha256).hexdigest()[:12]


processed = [
    {
        "subject_id": pseudonymize_email(u["email"]),
        "age": u["age"],
        # City is excluded because this analysis does not segment by geography.
    }
    for u in raw_users
]
pprint(processed)
avg_age = sum(u["age"] for u in raw_users) / len(raw_users)
summary = {
    "count": len(processed),
    "min_age": min(u["age"] for u in raw_users),
    "max_age": max(u["age"] for u in raw_users),
    "avg_age": round(avg_age, 1),
}
pprint(summary)

## 6. Guided exercises
1. Create an accountability record for the age-distribution workflow: state its purpose, proposed lawful basis, fields kept and excluded, retention rule, owner, and evidence needed before launch.
2. Handle an erasure request across records with different purposes and retention duties. Decide what to delete, retain with access limits, or escalate, and document the reason for each action.
3. Create a checklist to answer an access request for this dataset, including identity verification, data scope, timeline, delivery format, contact point, and evidence of completion.

These exercises move from implementing individual principles to documenting and defending an operational decision. Capture your answers in new cells before reading the solutions.


### 6.1 Solution — Accountability record
A useful record connects the business purpose to the data, legal assessment, controls, owner, and evidence. The lawful basis below is a proposal to validate, not a label that automatically makes the processing lawful.


In [ ]:
# Solution exercise 1: document the processing decision and its evidence.
accountability_record = {
    "purpose": "Produce an aggregate age distribution for product planning",
    "proposed_lawful_basis": (
        "Legitimate interests, subject to a documented necessity and balancing test"
    ),
    "data_kept": ["pseudonymous subject_id", "age"],
    "data_excluded": ["name", "email", "city", "marketing_opt_in"],
    "retention_rule": "Delete row-level data after the approved analysis window",
    "owner": "Product analytics lead",
    "required_evidence": [
        "Approved purpose and necessity assessment",
        "Legitimate-interests balancing test",
        "Field-level minimization review",
        "Retention owner and deletion evidence",
        "Access-control and pseudonymization design review",
    ],
    "release_status": "HOLD until the proposed basis and evidence are approved",
}
pprint(accountability_record)

### 6.2 Solution — Erasure decision under retention constraints
An erasure request does not produce one automatic action for every record. The controller must assess each purpose, lawful basis, and retention duty, limit access and further use when data must be preserved, and explain the result to the requester.


In [ ]:
# Solution exercise 2: decide record-level actions for an erasure request.
class ErasureRecord(TypedDict):
    category: str
    lawful_basis: str
    retention_required: bool
    active_purpose: bool


erasure_scope: list[ErasureRecord] = [
    {
        "category": "marketing_profile",
        "lawful_basis": "consent",
        "retention_required": False,
        "active_purpose": False,
    },
    {
        "category": "paid_invoice",
        "lawful_basis": "legal_obligation",
        "retention_required": True,
        "active_purpose": True,
    },
    {
        "category": "support_history",
        "lawful_basis": "legitimate_interests",
        "retention_required": False,
        "active_purpose": True,
    },
]


def decide_erasure_action(record: ErasureRecord) -> dict[str, str]:
    if record["retention_required"]:
        return {
            "action": "retain_with_access_limits",
            "reason": "A documented legal retention duty currently prevents deletion",
        }
    if record["lawful_basis"] == "consent" or not record["active_purpose"]:
        return {
            "action": "delete",
            "reason": "No remaining documented basis or active purpose justifies retention",
        }
    return {
        "action": "escalate_for_review",
        "reason": "Reassess necessity, the individual's objection, and applicable exceptions",
    }


erasure_decisions = [
    {**record, **decide_erasure_action(record)} for record in erasure_scope
]
pprint(erasure_decisions)

### 6.3 Solution — Access request checklist
- **Contact point**: dedicated privacy inbox (e.g., privacy@example.com) or DPO; log receipt of the request.
- **Identity verification**: request proportionate proof (avoid excessive collection) and note validation date.
- **Data scope**: identify all personal data linked to the requester, including pseudonymous records when they can be related back to that person; exclude secret keys and unrelated internal security information.
- **Timeline**: respond within 1 month; if complex, notify an extension (up to 2 additional months) with justification.
- **Format**: provide an accessible copy and the required processing information; use a structured, commonly used, machine-readable format when the separate portability right applies.
- **Logging**: record request date, scope provided, exceptions invoked, and closure date.
- **Follow-up rights**: remind about rectification or erasure (subject to retention duties) and the right to complain to the authority.
